# 🕵️ Dia 63 — Correlacionando fontes no Elasticsearch e Kibana

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 63: Correlating across sources in Elasticsearch and Kibana* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-63.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A TI da Serra Azul liberou uma exportação do SIEM do dia 5/out: 154 documentos, em ECS e UTC. Rafa: *"Faz uma pergunta só: `user.name : \"cx.centro07\"`."* E aparece, às 12h32, um login no webmail vindo de fora.

Neste notebook você vai:

1. **Conferir a ingestão** dos dois índices (390 e 154 documentos).
2. **Rodar um mini-KQL**: `campo : valor`, `and`/`or`/`not`, CIDR.
3. **Reproduzir as consultas ES|QL** e conferir as três linhas esperadas.
4. **Emular a EQL** `sequence by user.name with maxspan=30m`, com e sem `runs`.
5. **Desenhar o mapa de calor** e as tabelas do *dashboard*.
6. **Jogar o quiz "qual linguagem?"**.

🐍 Só biblioteca padrão. O Kibana real está na célula opcional.

## 1. Carregando os índices

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "lab:siem/lab-events.ndjson": """
eNrtXV1vG8mxfc+vEPwcEd3VVf0xT9fB3UXykDxcG8gXggtapGzCNKnLj/UVgvz31JASJXs1lMiusaY6bawXu6I0pE6dqak+XV3n
n2/+azP7Ml1vxl9u3jQXb8CAvzTu0qT3hhrAxpiRMeZvb3578ebTcr0ZLcZfpu03/vnd5c9/+NOlCe0r01+mi81oMt6M19NN++rX
2WKy/LoeradX29Vsc/vwTVfLye7n0QO2X92up6vDRd/99d37n/746JvHm+nH5eq2fW283Xzir834a7Pl4uF7ltvN1XL/4+vt1dV0
vW5f4w8wX34c8d/lYrS5vWlfJ/76fPxhOufPdbvgq/HF+Mub1Xb6r9/8sxsIaxpbgWiBcI1xDbgfCMT46oq/Yz5dfTa2LzjC+XC0
iITC4LD2fDzINmArPfZwYAOmvLsFMuBIfWZR6geO6/Fsvl1NxeGgxkS+X+rNcg8HhgZtvVn2cPg2d9hUHy0HPDA1zv9oPObT8cTA
4NgRGoMNpXqz3MPhfOPqzfKAB8bGhfpo2cPBWCCv4MrKHWejwTVYagBqJr1Dg0xDWDPpDg6wbdlBvmbSAx7uNW6W4dKDqzAbKhzu
0pr3hotS+I/XBRkIXto721hT08YdHry2N+UJgyEDjvgK9BhgAbZHA1wDvpLjHg7EBusz5Q4O394rVSV9gIOLjlpxPMDhfK04dkAQ
NY5qxXGHx04RNFgfKvdwAP3Y+2SoIkeLBq/ZUmOqeH6PBlp+yNbUsccDeKXCj5Xa1HGAA2ODrtKjxcO+t8CP2Vpy2F3zE69iqa5i
79Boa3Oot8kDHlx0pLLYkQUHZ1GoD5U9HNjWo4iVHfdwkGFE6qp+Dwc1EKrm8wgOdLXisDu1xzVENYke4EjlLdwy4OCFiq37CHdw
tBm0sbUgfcCD123FdZBm0IPoFTYShlqBxVYEq/r5AY/2wdKgqXfLAY7ymgQz0LCmodrOcYADbENVMH6Ag15BMB7ok6XtH00N1ifL
Dg9oT1Gbeni4BWK/fKtAWGyPDUM96naAg/Nncf1wGXDwQoVqL+09Hpw/bXkyRw4e1tZOn0dwQCiu+jobDc/LlFdoPB9ipw+0IoeF
uqh/BAe0RyFrJj3g4Vx5u5A5eLT7blSTxw6NtDv6V4f7POCBWOeVPIbD1z37ezjaQUemsuMAh20XLVD7zxkN996kpl3FdewmXK8f
x+8REOvb9Wb6ZdR+bonf6Bu0ljfr0XjyZbb7qfVyu7qajma7z23NiP/h9bfd4ci/0Xr88fBBL9brT5O/WxvjP5qLt3ztm810cnGz
/TCfXX2e3l5cL1cXh2tfXK+WXy4eXfDiZrnaXJBJ/J98IWgufvpvILLp4t3v3wL55v/iF/uX/zcMsjkgbPi65nq2+Dhd3axmi415
y3/enBmKfZvif7wItwOC14110sEBD+QbtLEdCfzDeN1+IiU36uHT7u9WsNGEjNvVEuJr3a67Eaxgqja6h8P/8GNdA81evp2W1zXx
SvvdajHjbgVjzWvdrbtDmFAbFA9wtOVGld8OcCCVJyicnctjC4ep2353cKTdLOl6JOIAB9QxGI/hcK9xZnmYekKLBpY3FOTcB0t7
gLt7TX/9taMOvJ6tpl/H8/nTv85iuvm6XH1+eHF8tfu99lhwZda+cvdNo81qvFi35Vf78ubqpqP6Q67UwqPX7n4CU4otFSb8280W
O/jufyzByIxgBObNd6/f/yi6Rx/jw+1mum6/Sh4Tf321nU8fIjifL79eftpsbtaXHJM354PdtpLXWbOP8EgayHdYenxLPjIGj5PP
nkY+axP5/sjnoC4wHsGBRjP3UkruKPfcqdxDtP1xj1D1U8ZbfxxsfxLYIaJ3PYIdlYDt7RPMJhuPgw2nge0cxr6wtu08Rr1Yo8NA
x59g8SSsIaYUegQ7aE7ZMYI9nkXsaWCbgNQf2EemBSpgNoZnFgYnPh8DGuoT7NDYOlbsAY8jw7GHf6fzqjQcv9PxNPJBAOyPe/ga
p1cHeQRtBwfxuiBVtewejVQ7HR7wgLYrXHFi8tGhpFzmAld8fWUmOGadqmAhE5M5Xlzv6f/ypwAvGnsEmzQX12hdOr5qtP5EsDl8
fYHt2npPMbODBSO5kok29Cc+uWMjwRQsG0PcLamPpJFwWs62Pe1w4HtjjrWIDl/oY6xjPE7sdKIeEsn0BzaoXqIH/wzYJ6ZsJIvY
H9i6l6Rgn3s+npZFgqHYG7PtzgPph4E9WS1vToM6xRFZhtuMwlPPR29CR5m9ixFwmDrAdm5XT3+Ptv8V1JPp9Xg731xOpovbHJxB
Cc4+PoGzJdKCs+qVTEr4TKZ2p2Vq6wL1mDy86q4C659RruHENXok1x/YOgSRrp1dH57ZAHMnrtEN2tgj2EFJuo7+SVm8awNsaOka
1OD8VLoG6Kr1hoaz09z/hYb8cRkETlstIqaeZJAd2K11kd7Hoo3oJDUnj3G3Q9wX2KR5aU5gjORjkUJ/mpNtyKreLI4BJYtrcAg9
1nukubjG6J/Zk3F0Wtsk19a+L7ChnR38Yw+ytMcjZ2sGebNc9XWSxT0dsvuy5HyoejzWURpUPY7y+jVU61+u/vfD+Orz9kYdTj2O
pSmLUr0eMXudE3e9IdXj3LjCkOrRBa6026/Hc2VFkQr6HAhcGFI9OvuWdftBn1O3CyqoXJ8Dhou6+VwDWG++F0KFtpLqZUhRrKR6
EVSaz6uSsRElVS5PAahHlQv79EYvjZeuroleChWmCtWLoOp1mlZB9Ss1UJdEL4UK687FS6GiunPxMqg8p/UK1UugQtUVbPThmXb1
Ew9iWADTXwWLSfVygeB4ey/QiR0IzvYIdtvuAQMGGwzXCCNr3QjpqX4PMPG87jyAJ3upu4i9Xn+6vJuQnIm2e61h0Y8y56/z8rPA
H58UTf9oLn7m67dzosfrNYM62Y2Jni1+Gc9nk4v2DS8ezYt+fPX9wOhdKHcDo/Pw7Xqk6WCzBV1s7pIaXo/Nq+Vyk0lm30Hm9tJH
6HtnJpIHaNeSSAl9vS76dm0LvB59H1byGQQOL8nG+3c6xmcvwGenms+AuvjsBsfn5Wp8NZ9m8jm+hM/7dzrCZ0ABPmPUzGdndPEZ
09D4/K3amsHp1MHphzc4QmVnsqkMnSsRJVQOmqgMnU7hr0dl/tUzK2UwL0nM7fsc43IQ4HJXt7oOLiPp4nLXdM3X4/L2w3ax2Way
2b6Ezft3OsJn/s98PkNSzeeki8/ODI3P15ub9v8zCQ0vIfTdWx1jdBJgtFNdOBMoY3QqUGUGJ6EyEwiwuesokxI2e11sJihPZQY8
T2WmfFXONUY1fb0qVc51brDqVpmBRFRmjwJ87tqGUsLnqIvPXWe+dKvM4EVUZh8F+Ayqi2WfdPEZilWZIeSozD4JUBlVlxpBWWpG
KFBljgIqc5BIy2RUc1lZWu5aYytXmZOIyhzyczMq7/yMqnYAcYCdnyIqszMyKnMMAozWvQ+YvC5GD28fUEBldlZCZU5egM1Oc+3s
jNPFZlegyuzgLJWZYydAX9RcXjhrddEXXYkqs3MSKjMHU4DPpJvPyooLciWqzA4lVGYn0JtPuhtAHajKzzTABlApldlRhsrMcRSg
suoNQKerLZ8GuAGYrzI7n68yO4G+fOo8lKaEy0EXl4FKVJldkFCZnUBvPjXOa+YzOl18dqFMlTmKqMwcTgFGY1LNaGUZmkyJKnMS
UJk5lNls9rqbQR2q6tDwA2wGzVeZ0ZynMmMUoK9VrWJQ0EVfW+TEDLQiKjNJpGNQXS5T0sVnCCWqzAgiKjMlAT6rbgB1Xll5MbwG
UCmVGV2OyuwlSg1SXWoE1EVlKnBiBqKAyhzyj0wFzYa6IZCXtIqOFDH2OGI1dBr1KEkcqjoHQqd5om5JH0lE0g9eIHmA6ppO10EI
RhuKlPTRy0j6Akchgu45lS46XYwe3pxKAUkfg4SkH50Am1F1C4Gugz2MdoFDsDGeJ+kLnOIJncZ6OuibSBd9KRUp6ScRST/lT/OL
jTWq+axKAmW0bYmSPhkRST9FAT6D5m5b1HUuLXZuCOqX9MlmSPoocEYtdrbL6aCyNbqo7GJ5kj5BvqTPgRTgsmpvAtR13jIO0JtA
QmUmJ6Eyo8B5y6R7PAlaVSpzKnU8CaGIyszhFGC06l5FBNDFaFuiykwSVoscSgE2g+raWdehy9Q5E1SzykznWS2iwDnLpHs8CTrU
Rd8yx5OQiNUiB1OAz6q7E9F5XXymIq0WScRqkYOZx2fXGNPZxqWDz6invNijXewQbMqxWuQ4ClBZtaszKjoCv0e7wMZxL2C1iLmH
33foOt1pOejisitSZfYiVouYe/x9hzCqVjFIWZmBsUiV2ctYLXI4sxltdQ90QIqaGG0HONBBQGX2ElaLHEoBNlvN5+HRW11stqE8
ldmfZ7XIsROgr2orLwxGF32hyF5mL2K1yMEU4LPu5V9QVlwMb/knoTJ7EatFDBLlheqh7hi9Lj4Pb6i7lMrsc6wWOY7ZVAbdUwYw
qaqUYYBTBgRUZgGrRQ6kAJdBdVpOSReXwRWpMotYLXIwBfjsNB+ZIoO6+Ox8kSpzkLFa5HAKMFq1GS5ZZdXG8MxwBVTmIGG1SDa/
3nC6z5qQBU1sdgM8a5KvMofzrBY5dgL0tbrpG3TR1xbZyxxErBY5mAJ8Vn20hAB18Xl4R0skVOYgYrXIwRTgs0uq+Uy6+IymVJU5
5FgtchwFqEyaZ8uRU5aaicpTmYOA1SK5/LSMnZ6sSrisSmVmtIu0WgwiVoscTAE+q+6VI129+TjAXjkZlVnGapEEuvOxcapVZvK6
GO2KVJklrBY5lAJsRtW7gLp6mRltX57KHM+zWiSBXmbsnBSshL6qVAzqdOjRrTJHEatFDmY2n0n3OC7yURefbZETM6KI1SL5KMBn
1U1zFLwuPkOxvcwxx2qRghegsrOqqRx0UdlBeSpzFLBa5EAKcBlVlxkRdHEZsUSVOYq4/3EwBfisewcwJl18Ht4OoIjKHGXc/zic
2Yz2nTtTOhidVM3nYrR9gSpzlHD/41AKsBlU75mkqIvNYAtUmc9z/6MUBeir2orVG9BF3+FZsYooGCLufxxMAT6rHjDnDeniMxbZ
y5xE3P84mAJ8Js1NRt6iLj5TKFVlTjnufxzHbCqHzpWIEiqrUplD53wSzSpzEnD/8wJHpkJnN6IOLus6YsJoxxJV5iTi/ucFzpmE
zlF+OvjsnC4+O1Okypxk3P84nAKMVj060TuvjNGpQJU5Sbj/eQF7nqB7/ovXZc8TBjj/JV9lTue5/3kBS57Y2V2rhL6qln6x05JA
t8qcRNz/vMBBqdjp36WEz0kXn60pUmUWcf/jYArw2aoulskq43Ox7n8px/2P4yhAZdXDjDyRLipDeXOZkxFw/+NACnAZrWouKyub
EQpUmZMRcf/jYArwWbVpttfllRYHaJotoTInI+P+5wX80pLuSbaeVC0E0wAn2earzMlIuP9xKAXYrPqItveoi83DO6KdrTInc577
H8dOgL6AqulLuugLVKDKnIyI+58XmJiROo01dPA5OF18dqFAlTkZEfc/DqYAn3VvYkeri8/D28QWUpmTyXH/4zhKUFn1BqCqU9nW
DHA4l4DKLOD+5/PPY7foquZy8rq4PLzNbBGVWcT9j4MpwGfVm9nBOGV8TkWqzFbG/Y/DKcBop1llDqr80lq0S1SZrYT7X8i3T2N8
0atms7L8jKE8ldme5/7Hscumr+3srtVBX9DUit+iDSWqzFbE/Y+DKcBnq5vPQRefLZSoMlsR9z8OpgCfQTWfVVmmtWhDqSqzzXH/
C/mOaQyu0zwpMSDoorLz5anMVsD9jwMpwGVUrcqhsrSMqUSV2Yq4/3Ews/kMnb2ISvisqdmoRbvMXmYr4/7H4RRgtFWtMqs6BNii
XaTKLOH+F/LPATK+oJvNThebh2fPk68yw3nufxw7Afqq7v0MpKy8GF7vp4SCASLuf4Ekigvdm36qDkq1aBfZywwi7n8h/6iUdbon
GgWvahfQDXCikZTKDDnufxxHASpbVE1lr4vKlspTmUHA/Y8DKcBlUK3KBdTFZSjR/S+BiPsfB1OAz7p3AKPTxefh7QCKqMwg4/7H
4cxmNGp2gSBjQdWeNr6iC8R6y/Cv1/0Vzu1e4NtdiE8vnXeRzCZzGnav0bek/C452+OFs+N/h1O43FpAYzeZZ4vNdLUYzzPh7tL0
18Mi8rfIP6Jx+0H3DHYGdy35DwzefpjPrj5Pb5+k8MMV77Kxva+Xm4v/eff24t3v3wL55m8fwgFWwzgbvs7+MoavbH53+JMRBwid
c+n+/O7y5z/86dKEJ4PxdbaYLL/yG06vtqvZ5vZRRJaT3c+jB3wG59wQ8oeYLz+O+C8TenN7077uno7f4R4YIlABngHqzA+N5gdK
W5PV8ua0lJbiiCwHx4x8fOIBDQnPLDmd2x2K/j6t+V89oCfT6/F2vrmcTBe3OTjb4UiI20lXCjvcAt/h7PBoIcRId8C829f8HuQE
Tz84Jou1xIMD7bDHtB2F2iXqMtF7uBfAdcCN+BTeGMG6ZDoe1582m5v1JWfPDMhJSxZJ5qkyP5x7qPxHZxGiYW9E7AEj/jc8UYNS
7BpEkaDNISObTuK1IwepL1L7Brt3fXopfN799d37n/7YV9FDZwJh4djdXRoQ/wY0L48b""",
    "br:dia-63/sa-eventos.ndjson": """
eNrtXW1v28gR/t5fEfhTC1S63dnZN37qIU37KdeiSdEXCDgwMuMIkUWVpGr7Dv3vXUp24rO59IriUNwcAQcItJS4nOeZ4ezM7OzP
F3+oVtdZWaXX24vk1QUwUDPOZky+5yxh9d+cMfbvi9+/uviUl9V8k15n9YXbIr+9m5VpPZD9N9tU88u0Ssus+jL4dWSZVtlVXtzV
QzfZh3qgzHfFMpuv9jflbA5sjmwueT22K7Piy32Wt/Ol+40iZ3I/VqznebG6Wm3SdT18nW7Tcp7duumvswRR7CdaVdt5kf1n555r
fp1Vn/LL+trXf/nhhzev3z+6otzmmzKbu6evduWPy/yyviUw5q64dN91N6lW+eZhlhbmbqJzzh89wIe7KivrLykhn3zrYQjRauHG
1umHbF3Oy7tN9SmrVks3VBW77H+/+bkrBh9vfAB8XBXZTbpeN2OwyaqbvPj8dTBd1jOuh9LlMttW9cj9RfOqSDdOTsX+d6vl9iX0
7sfuvyEltzJAnI/H77/qwHw0jS/S1IoZ93mxW2dfBOGeNL+Z1aCWs3y3n35XaSv3NyTj4Rnjr9z/7ynPR0J5YRsoLwwaH+WVloYE
hLFRHhooj0zZAHEGU14yxRQd5es/GJLyosXIqybGrzY1Ilk1W15uBia+biA+19p6iK+Z4JwEirERXzQQ3wilAsQZTHwNzDJC4uPA
3k0ctr6J8kIz9Nl6rSWSgBCBrXeuHbd9Ul5ylITuDfAExZCUl88on1aXD5SHJsp/zItNtswu82K2TIsqz4bmv2py79mzbz0MKe7M
HgkiY+O/bDD5FoPEGcx/JQ7SpOK/TOSgJl+3+Dq6if61dNL1bJO7h1oNbfsBm2w/Cq+7g0YwEjTGxn3dyH2wAeIMd3cM7BcOVNy3
CeCo/fxzm36AZm+fe+gPiitFAkgM3r4wlgeIM5j+oK3QdPQXOHb6n8HZB9bEeBA+Zwc4cEGCQQyMd76ODRBnOOPh8DkV400CetTO
zjmi96LRxdG+5a1wDqkmwSAGF0eIZ+v+JnEGM17oQ3yMiPFoEsZHbeM/7MplOganXigf47lmRpJgEIONV5LrPp16YIiETr3kCfNE
jN99P3v9z9nB8D4T+M1qc5nfuPtly12xqh6T/8CqC1SHJ/0Fu9/96937N29b5tuIX7pz12zcRXt5fb2Ve/RlfvjhcueALMt6zE1t
nV/N3T8n2+puW4/LADnguDPVZ49r2ea0tfYublCqk4iJEaetDd/7ub3l8MAyS+jqSUg4G84MPHHrhrUFECAMwYmEYUyfwnDW5+FJ
7//rbIK7e5V+WO9/+3WyWPzjMK3FIrvdrvPC3dld8/gr27Sof7jlm+/uyiq7FrBY1DNfbVb7qy9eFKTwvlyccbxOV2ufIt8PN6qq
I0AN6GrzkiTb/NQ20pxAxwCJIPt2qPXXIr8q0utXf1qts/LVb2+N+t1i8Xa1LPIy/1gtFm8ur7LF4vvtdn0voMXiuszch8dT8Bfk
fVHMmCAfkHhP07BtCcBzMU8maMedrD5fmQay5sCt8OXsBLOn+DZ+LGKoT2LchogzPGeHhi5v4dY3MmE4Md+zGm3KVgtrfV69Bm4Y
CRYxMF9ZoQLEGZ6xU87I0DGfiwTklK32giWbuC/Rx30l5d5O9Y9GDKFczYLEGW71LeeKkPsmQT3uSqWzkr9xF4Lg4HN5tFDaksAR
Q5mS0Fz1uQtBK8OAkPw2EWbUhr9+yuS779abzw+0/65MdQvt//zmRcqLvUibMDJzyeec1dNtoj1y8GXvTnF2WlCIweBzxk2gPBt4
vxfcM9rbFtKfznnko+a8u7RI059269l2dTuMtQcmnHniXMx1404Eg75KDTQcGAkacXAfIEig4fvOrFWMzuJDvbaa2H8U++FQS9DE
fnkS+VvAiIP8gvdLfo2Ck3JfwsT947gP1mv5rWEkYETCfdUz90FT2n0x8ObLb8HrUeDz991KwJzEfhHNBsxm9iPr2e3hKClXuoDf
VK7+7+5m5WLx6Db79Okf3cQXi7/l6fVqc7VYvH33fZ1ITWdOGdL16qf0Mi9IM6p7OSN+gzURXysbyuUnN+vPZbAgT8TqRYlLb/HE
SCz7wxPNrvPL3TofKn75JfjAm9O27gpfDLN70rYVjyhMOw8WaLBx59YioWlXAyvA8RH8c1RjN3rz2nq3WMKhfrh/EGKI24NFDBBn
uDMvAQ0h5W0icbL5x9t8AO0v1SHBIxKbD73bfCMJKxYEDlyldnQZ/lmTtqKxYoFZ31YcKdkpxr8FjRhq8BWXJkCc4b1VDKfbbrmX
tlST8T/e+Ev0Fi0gJ8EjDuPPed/GH8AQOvxi6AxuRI3keGNXIfRWKKM+JYwp4sneNu3BtAxVgDSPCOFLQUt7Pdn9Dk6/v42i5iR4
xGH3ofdADzBJmMNCkUg5dYkOMfjW+CoWDFpGAkEEnj5aaXs1+JyjJixYQOdZ8sniH2/xBfN6+vuUbv94RGLxZf8Wf/+bVApgEmam
XlpPQtFNnYX4oZdKc+NQcUpFfgsGUfj4YCFAnOHbUQAV4doW7dgbh441uAM+/humSPCIxOTb3k2+oCzSlO4daycF6KAAbXtxSfCI
QwEE9K8AnDC3K/XACnD8NvQzlDNwaIxrek+GqcszLQkIcew/1xAgzvB2im7lQJjNlXbshfkjtfmofTbfgCLBIxKbT+D0MLLQPiQM
E5STAnQI9Hj3oRtrSfCIQwFQ95/SBSBUAO3tKzgpQIsCCI5Sg88Jut+d2j8mUSiB0L0rgeCSAWeEemATnDyhLsX8TPnCn2BJ8Ijj
RSCx98JOLSSdAnBMhJxWv0EHxqBh3m3pyEgwiOKoPIXQ53kxSgEjJTxO8c5OVT3eeCdjJHhEYvF1/9u36JoPOoHrqeHssdU90lfQ
JjRyS4JFHIekMtlneQ8yoQ0h820izWT6O0R9JPOafkuCRxymX3GCgk5LpwAw5Xo75nr9xT6gSfCIRAFU/2FPThj2BJUINW7f5+xn
pWJjtZv2vQEEoLYkiMSQ+AWjez0qWAiJhC8AwRImphdAh8SvBW//ZUaCRyQvABtX3ssJXMC0j9278bqJ/OAv9ZQSgQSMGLaxC6N5
gDSDmW+YIgz7CD7yhjFfW9nZJmmDVp7Uyh4keewZzYr7oswPJ1EdzpPqX9yDH7H2VLLPTrr66O63KzLSk672IuEyYgYqwMgY6BP3
r5iB4z67tJ2BqGVsDPSJ+1fLQBh5B/KX3sLga17SjYHGoC/e0w8F/fIeHwWHOHDSiUR4d1h1F0n9+Yf8dl7j+OOyyNz0hpbNi88t
x34Sw2hLvby9DfbVTv3jEUfsQxMk/iVhqZdQCePT5u4nGWrTeAiDt7pR4GF/c/8YxLC522gTIs7wcLdCS5jwRzZwO4NvJtwNvnSP
MpwEjzhMvum9uhcOb1cqBZgS/h19Hu4tb0dLgkccCmB5/+XtitDnkc7iTD38+u1uIDkJHlEoAPR/WAMINKQKAHzchzWcveKlqb+T
sN4lAIJkSIJIDCc3aPlsC/BJ/Z1QIeUGPzlFfbq2OgDvyQ2cBI9IXgCCYAlA1upAJIwlfCr56tTgTPUf9mzFIxIFIOj1YRihAoiB
z+453gM6Q9xTqMY9fswX9wGmu/cxbgUhAqdHMeAYIM4jTL61lJSX3igzSdbzaU17WwH1WRLve5HwKRLWJRKGvuyf5uokivKYI2HA
RUwneLUKfHw2YYhKCJFw54pNwcEu+0EF964NT3CNW/CIxCbY/isC6KLjTuDSeyr6yYdHg+zz8Ognmv2CH+Hm54xSbZicfKu77Z7t
AcIARiUMHFgYjwxoZ2FMUYNOiUNfqwij5UmaClFHDUDEdMi1cKYggal0pEvcWHlLR7onDlvxiEMBRP+9Ig4HSVIpAE+EHnW14Lnz
hqLpxHduvUc/SnZC3rAVkBhKB9EoESDO8EN/ATXhchncC3daG3baKc+858IACR6R2H+CnfIWCBXADGz/j94pf8YucbaJ+ca0dInT
nASKGPbJc85EgDjDi8atUITJk7orwVQx0mXp6z8cw1gSPOKw/Nh/kyy2b7xFpQBy4B4p30yHXG/RuAZGgkccCiD7Lxp3rwA6BUDm
3bc+KUCrAjDsv01iKx6RKED/eSEjCGM/CAkjSoUYxjqmQl6cNI7+HL+z9vaCxnAVl94T/azQigSOGBo7ogQIEGf4ooVJQZiukHbg
V9bR0drzrdahabUujPLVt3PUxpBAEUOcVjFtAsR5xJtKyZNW6/8Hh9zOuQ==""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
import json
LAB = [json.loads(l) for l in carregar("lab:siem/lab-events.ndjson").splitlines() if l.strip()]
ISCA = [json.loads(l) for l in carregar("br:dia-63/sa-eventos.ndjson").splitlines() if l.strip()]
INDICES = {"lab-p4": LAB, "isca-sa": ISCA}                 # nossos "índices"

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

In [ ]:
# Passo 3: conferência de ingestão ANTES de qualquer análise.
from collections import Counter
for nome, docs in INDICES.items():
    print(f"=== {nome}: {len(docs)} documentos")
    print("   datasets:", dict(Counter(d["event.dataset"] for d in docs)))
    print("   hosts   :", dict(Counter(d["host.name"] for d in docs)))
    print("   tempo   :", min(d["@timestamp"] for d in docs), "→", max(d["@timestamp"] for d in docs))
    print("   sintético em todos?", all(d.get("labels.synthetic") for d in docs))

## 2. Mini-KQL

Suporta: `campo : "valor"`, `campo : valor`, `campo : 10.0.0.0/8` (CIDR em campos de IP), `campo : *trecho*`, `and`, `or`, `not` e parênteses. É um subconjunto, mas as consultas do dia cabem nele.

In [ ]:
import ipaddress, re

TOKEN = re.compile(r'\s*(\(|\)|\band\b|\bor\b|\bnot\b|[\w.@]+\s*:\s*(?:"[^"]*"|\S+))', re.I)

def kql(consulta, docs):
    """Converte a KQL num predicado Python e filtra os documentos."""
    tokens = [t.strip() for t in TOKEN.findall(consulta)]
    pos = 0

    def termo(tok):
        campo, valor = [x.strip() for x in tok.split(":", 1)]
        valor = valor.strip('"')
        def f(d):
            v = d.get(campo)
            if v is None:
                return False
            v = str(v)
            if "/" in valor and campo.endswith(".ip"):                       # CIDR
                try:
                    return ipaddress.ip_address(v) in ipaddress.ip_network(valor)
                except ValueError:
                    return False
            if valor.startswith("*") and valor.endswith("*"):                # curinga
                return valor.strip("*").lower() in v.lower()
            return v.lower() == valor.lower()
        return f

    def expr():                                                              # or tem precedência menor
        nonlocal pos
        esq = conj()
        while pos < len(tokens) and tokens[pos].lower() == "or":
            pos += 1; dir_ = conj(); esq = (lambda a, b: lambda d: a(d) or b(d))(esq, dir_)
        return esq

    def conj():
        nonlocal pos
        esq = unario()
        while pos < len(tokens) and tokens[pos].lower() == "and":
            pos += 1; dir_ = unario(); esq = (lambda a, b: lambda d: a(d) and b(d))(esq, dir_)
        return esq

    def unario():
        nonlocal pos
        tok = tokens[pos]; pos += 1
        if tok.lower() == "not":
            f = unario(); return lambda d: not f(d)
        if tok == "(":
            f = expr(); pos += 1; return f                                   # pula o ')'
        return termo(tok)

    pred = expr()
    return sorted((d for d in docs if pred(d)), key=lambda d: d["@timestamp"])

CONSULTAS_SALVAS = {}

def rodar_kql(nome, consulta, indice, mostrar=6):
    res = kql(consulta, INDICES[indice])
    CONSULTAS_SALVAS[nome] = (indice, consulta, len(res))                    # "Save query"
    print(f"\n[{indice}] {consulta}\n→ {len(res)} documentos")
    for d in res[:mostrar]:
        print(f"   {d['@timestamp'][11:19]} {d['host.name']:10s} {str(d.get('user.name','-')):13s} "
              f"{str(d.get('source.ip','-')):14s} {d.get('event.outcome', d.get('event.action', ''))}")
    return res

rodar_kql("falhas do 203.0.113.45", 'event.dataset : "system.auth" and event.outcome : "failure" and source.ip : "203.0.113.45"', "lab-p4", 2)
cadeia = rodar_kql("svc_backup com sucesso", 'user.name : "svc_backup" and event.outcome : "success"', "lab-p4")
rodar_kql("198.51.100.23", 'source.ip : "198.51.100.23" or destination.ip : "198.51.100.23"', "lab-p4")
rodar_kql("203.0.113.0/24", 'source.ip : "203.0.113.0/24" or destination.ip : "203.0.113.0/24"', "lab-p4", 2)

In [ ]:
# A cadeia do passo 4, com o campo que liga cada salto.
print("Cadeia svc_backup (LAB-P4):")
anterior = None
for d in cadeia:
    salto = f"de {d.get('source.ip')}" + (f" (logon tipo {d['winlog.logon.type']})" if d.get("winlog.logon.type") else "")
    print(f"  {d['@timestamp'][11:19]}Z  {d['host.name']:10s} {salto}")
print("➡️  O salto fs01 → WS-FIN-07 (03:27:41) não estava na história do Dia 58.")

## 3. ES|QL em Python: `STATS ... BY ...`

In [ ]:
from collections import defaultdict

def esql_falhas(docs):
    """FROM idx | WHERE authentication AND failure | STATS COUNT, COUNT_DISTINCT(user), MIN, MAX BY source.ip, host.name"""
    g = defaultdict(lambda: {"failures": 0, "users": set(), "first": None, "last": None})
    for d in docs:
        if d.get("event.category") == "authentication" and d.get("event.outcome") == "failure":
            r = g[(d.get("source.ip"), d["host.name"])]
            r["failures"] += 1
            r["users"].add(d.get("user.name"))
            r["first"] = min(r["first"] or d["@timestamp"], d["@timestamp"])
            r["last"] = max(r["last"] or d["@timestamp"], d["@timestamp"])
    linhas = sorted(g.items(), key=lambda kv: -kv[1]["failures"])               # SORT failures DESC
    print(f"{'source.ip':15s} {'host.name':10s} {'failures':>8s} {'users':>5s}  first → last")
    for (ip, host), r in linhas:
        print(f"{str(ip or '(vazio)'):15s} {host:10s} {r['failures']:8d} {len(r['users']):5d}  {r['first'][:19]} → {r['last'][11:19]}")
    return linhas

print("lab-p4:"); falhas_lab = esql_falhas(LAB)
ESPERADO = [("203.0.113.45", "bastion01", 112, 8), ("10.10.20.5", "WS-FIN-07", 23, 3), (None, "WS-FIN-07", 5, 1)]
obtido = [(k[0], k[1], v["failures"], len(v["users"])) for k, v in falhas_lab]
print("bate com as 3 linhas esperadas?", "✅" if obtido == ESPERADO else f"❌ {obtido}")
print("\nisca-sa:"); esql_falhas(ISCA)

In [ ]:
def esql_bytes(docs, campo_bytes="network.bytes"):
    """WHERE firewall AND accept AND NOT CIDR_MATCH(destination.ip, "10.0.0.0/8") | STATS SUM, COUNT BY src, dst"""
    rede10 = ipaddress.ip_network("10.0.0.0/8")
    g = defaultdict(lambda: [0, 0])
    for d in docs:
        if d.get("event.dataset") == "firewall" and d.get("event.action") == "accept" \
           and ipaddress.ip_address(d["destination.ip"]) not in rede10:          # aqui o tipo 'ip' importa!
            g[(d["source.ip"], d["destination.ip"])][0] += d.get(campo_bytes, 0)
            g[(d["source.ip"], d["destination.ip"])][1] += 1
    for (s, t), (b, n) in sorted(g.items(), key=lambda kv: -kv[1][0])[:5]:
        print(f"{s:15s} → {t:15s} {b:>12,d} bytes  {n:3d} conexões".replace(",", "."))

print("lab-p4:"); esql_bytes(LAB)
print("\nisca-sa:"); esql_bytes(ISCA)

> 💡 **Por que `destination.ip` precisa ser do tipo `ip`?** Se o campo for mapeado como texto, "10.0.0.0/8" é só uma *string*: `CIDR_MATCH` não sabe que `10.10.30.17` está dentro dela. No Python acima, quem faz esse papel é `ipaddress`.

## 4. EQL: `sequence by user.name with maxspan=30m`

Emulação: para cada usuário, uma falha seguida de um sucesso em até 30 minutos. Com `runs=N`, exige N falhas seguidas antes do sucesso.

In [ ]:
from datetime import datetime, timedelta

def ts(d):
    return datetime.fromisoformat(d["@timestamp"].replace("Z", "+00:00"))

def eql_sequencia(docs, runs=1, maxspan=timedelta(minutes=30)):
    por_usuario = defaultdict(list)
    for d in sorted(docs, key=ts):
        if d.get("event.category") == "authentication" and d.get("user.name"):
            por_usuario[d["user.name"]].append(d)
    achados = []
    for usuario, evs in por_usuario.items():
        falhas = []                                                          # falhas consecutivas pendentes
        for d in evs:
            if d.get("event.outcome") == "failure":
                falhas.append(d)
            elif d.get("event.outcome") == "success" and len(falhas) >= runs:
                inicio = falhas[-runs]
                if ts(d) - ts(inicio) <= maxspan:
                    achados.append((usuario, falhas[-runs:], d))
                falhas = []
            else:
                falhas = []
    return achados

def mostrar_seq(achados, titulo):
    print(f"\n{titulo}: {len(achados)} sequência(s)")
    for usuario, fs, ok in achados:
        print(f"  {usuario:13s} {len(fs)} falha(s) a partir de {fs[0].get('source.ip') or '(local)'} em {fs[0]['host.name']} "
              f"{fs[0]['@timestamp'][11:19]} → sucesso {ok['@timestamp'][11:19]} em {ok['host.name']}")

mostrar_seq(eql_sequencia(LAB), "lab-p4, sem aperto")
mostrar_seq(eql_sequencia(LAB, runs=5), "lab-p4, runs=5")
mostrar_seq(eql_sequencia(ISCA), "isca-sa, sem aperto")
mostrar_seq(eql_sequencia(ISCA, runs=3), "isca-sa, runs=3")

### 🧩 Exercício

1. No LAB-P4 sem aperto, quem é o falso positivo? Repare que, sem aperto, a sequência do `svc_backup` mostra só **uma** falha (a última do *spray* contra essa conta): a regra ingênua não distingue um *spray* de um erro de digitação. Com `runs=5`, o que mudou na linha dele?
2. Na Isca, quais dois falsos positivos somem com `runs=3`?

## 5. O *dashboard* em texto

In [ ]:
#@title Índice e janela do dashboard { display-mode: "form" }
indice = "lab-p4"  #@param ["lab-p4", "isca-sa"]
inicio = "2026-03-13T12:00"  #@param {type:"string"}
fim = "2026-03-14T06:00"  #@param {type:"string"}

docs = [d for d in INDICES[indice] if inicio <= d["@timestamp"][:16] <= fim]
if not docs:
    print("Nada na janela. Para a Isca use 2026-10-05T10:00 a 2026-10-05T14:00.")
else:
    # Mapa de calor: falhas por hora × host
    horas = sorted({d["@timestamp"][:13] for d in docs})
    hosts = sorted({d["host.name"] for d in docs})
    cal = Counter((d["@timestamp"][:13], d["host.name"]) for d in docs if d.get("event.outcome") == "failure")
    print("Mapa de calor: falhas por hora (UTC)")
    print(" " * 14 + " ".join(h[11:13] for h in horas))
    for host in hosts:
        celulas = []
        for h in horas:
            n = cal[(h, host)]
            celulas.append(" ." if n == 0 else " ░" if n < 5 else " ▒" if n < 20 else " █")
        print(f"{host:13s}" + "".join(celulas))
    # Barras: bytes por destino (firewall)
    print("\nBytes por destination.ip (firewall), top 5")
    b = Counter()
    for d in docs:
        if d.get("event.dataset") == "firewall":
            b[d["destination.ip"]] += d.get("network.bytes", 0)
    maior = max(b.values()) if b else 1
    for ip, n in b.most_common(5):
        print(f"{ip:15s} {'█' * max(1, int(40 * n / maior))} {n:,}".replace(",", "."))

In [ ]:
# A tabela do dashboard + a cadeia da Isca
conta = "cx.centro07"
print(f"Tabela: user.name : \"{conta}\" (fora o proxy, para caber)")
for d in kql(f'user.name : "{conta}" and not event.dataset : "proxy"', ISCA):
    print(f"  {d['@timestamp'][11:19]}Z {d['host.name']:10s} {str(d.get('source.ip','-')):14s} "
          f"{d.get('event.action', d.get('event.code',''))} {d.get('event.outcome','')}")
print("\n➡️  Liga o notebook (10.20.40.57) ao login de 203.0.113.91 APENAS o nome da conta: inferido.")

## 6. 🎮 Quiz: qual linguagem?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True
PERGUNTAS = [
    ("'Quantas falhas por IP de origem e host, com primeiro e último horário?'",
     {"a": "KQL", "b": "ES|QL (STATS ... BY ...)", "c": "EQL"}, "b", "Agregação é trabalho do ES|QL."),
    ("'Três falhas e depois um sucesso, da mesma conta, em 30 minutos?'",
     {"a": "KQL", "b": "ES|QL", "c": "EQL (sequence by user.name with maxspan=30m)"}, "c", "Ordem importa: sequência."),
    ("'Mostre todos os eventos da conta cx.centro07'",
     {"a": "KQL: user.name : \"cx.centro07\"", "b": "EQL", "c": "nenhuma"}, "a", "Filtro simples, por entidade."),
    ("O dashboard mostra uma hora sem nenhum evento do proxy. Você conclui:",
     {"a": "ninguém navegou", "b": "nada aconteceu OU o coletor parou: confira a ingestão", "c": "o proxy foi atacado"},
     "b", "Silêncio pode ser lacuna de ingestão."),
    ("CIDR_MATCH(destination.ip, \"10.0.0.0/8\") não funciona. A causa mais provável:",
     {"a": "destination.ip mapeado como texto, não como ip", "b": "falta de memória", "c": "UTC"},
     "a", "É o item 'sem notas' do checkpoint do original."),
    ("A EQL devolveu ger.centro01 (1 falha, 1 sucesso, de dentro da agência). Isso é:",
     {"a": "conta comprometida", "b": "provável erro de digitação: falso positivo da regra ingênua", "c": "ataque interno"},
     "b", "Aperte a regra (runs) e registre antes/depois."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 7. (Opcional) A Elastic de verdade, na sua VM

In [ ]:
!curl -fsSL https://elastic.co/start-local | sh
print("Depois: Kibana > Upload a file > lab-events.ndjson (índice lab-p4) e sa-eventos.ndjson (índice isca-sa).")

In [ ]:
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
linhas = ["# Planilha de resultados (Dia 63)", "", "| Nome | Índice | Consulta | Docs |", "|---|---|---|---|"]
linhas += [f"| {n} | {i} | `{q}` | {c} |" for n, (i, q, c) in CONSULTAS_SALVAS.items()]
linhas += ["", f"EQL lab-p4 sem aperto: {len(eql_sequencia(LAB))}; runs=5: {len(eql_sequencia(LAB, runs=5))}",
           f"EQL isca-sa sem aperto: {len(eql_sequencia(ISCA))}; runs=3: {len(eql_sequencia(ISCA, runs=3))}",
           "Falsos positivos: acct.clerk01 (LAB-P4); ger.centro01 e cx.centro07 local às 13:15 (Isca)",
           "Salto fs01 → WS-FIN-07 (03:27:41Z) ausente da história do Dia 58."]
Path("notas/resultados-dia-63.md").write_text("\n".join(linhas) + "\n", encoding="utf-8")
print("\n".join(linhas))
try:
    from google.colab import files
    files.download("notas/resultados-dia-63.md")
except ImportError:
    print("(Fora do Colab: notas/resultados-dia-63.md ficou na pasta atual.)")

## ✅ Antes de fechar o Dia 63

- [ ] Ingestão conferida: 390 (LAB-P4) e 154 (Isca), hosts e datasets
- [ ] ES|QL de falhas bate com as três linhas esperadas
- [ ] EQL antes/depois do aperto, falsos positivos nomeados
- [ ] Cadeias do `svc_backup` e da `cx.centro07`, com o salto inferido marcado
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 64):** o relatório. Dezesseis dias viram cinco achados, cada um com um termo de confiança e uma citação que um revisor encontra em um minuto.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>